In [1]:
import pandas as pd
import numpy as np
import random
import torch
import transformers
import torch.nn as nn
from transformers import AutoModel, BertTokenizer, BertForSequenceClassification, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_metric, Dataset
from sklearn.metrics import classification_report, f1_score

/Users/vladimirkalajcidi/anaconda3/envs/second/lib/python3.11/site-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/Users/vladimirkalajcidi/anaconda3/envs/second/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/Users/vladimirkalajcidi/anaconda3/envs/second/lib/python3.11/site-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [2]:
from transformers import AutoModel, BertTokenizer, BertForSequenceClassification, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_metric, Dataset
from sklearn.metrics import classification_report, f1_score

In [3]:
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler,random_split

In [4]:
df = pd.read_excel('menu_pr.xlsx')
df = df.sample(frac=1).reset_index(drop=True)
train_text = df['text'].astype('str').to_numpy()[0:439]
train_labels = df['target'].to_numpy()[0:439]
test_text = df['text'].astype('str').to_numpy()[420:439]
test_labels = df['target'].to_numpy()[420:439]

In [5]:
df

,target,text
0,2,Какой ужин тебе приятнее всего?
1,0,Что можно приготовить на утро?
2,0,Какой завтрак тебе хотелось бы?
3,1,Какие блюда рекомендуются для середины дня?
4,3,Какие варианты еды доступны в настоящий момент?
...,...,...
433,2,Какие варианты ужина доступны?
434,0,Что приготовим на завтрак?
435,3,Какие блюда предлагаются в меню на данный момент?
436,2,Какие варианты ужина есть?


In [6]:
def seed_all(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.benchmark = True
        torch.backends.cudnn.deterministic = False
seed_all(42)

In [7]:
device = 'mps'
model_name = "DeepPavlov/rubert-base-cased-sentence" 
model = BertForSequenceClassification.from_pretrained(model_name)
tokenizer = BertTokenizer.from_pretrained(model_name)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased-sentence and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
model.classifier = nn.Linear(in_features=768, out_features=4, bias=True)

In [9]:
seq_len_test = [len(str(i).split()) for i in df['text']]
max_seq_len = max(seq_len_test)

In [10]:
tokens_train = tokenizer.batch_encode_plus(
    train_text,
    max_length = max_seq_len,
    padding = 'max_length',
    truncation = True
)
tokens_test = tokenizer.batch_encode_plus(
    test_text,
    max_length = max_seq_len,
    padding = 'max_length',
    truncation = True
)

In [11]:
class Data(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor([self.labels[idx]])
        return item
        
    def __len__(self):
        return len(self.labels)
    
train_dataset = Data(tokens_train, train_labels)
test_dataset = Data(tokens_test, test_labels)

In [12]:
training_args = TrainingArguments(
    output_dir = './results', #Выходной каталог
    num_train_epochs = 50, #Кол-во эпох для обучения
    learning_rate = 1e-5, #Скорость обучения
    evaluation_strategy ='epoch', #Валидация после каждой эпохи (можно сделать после конкретного кол-ва шагов)
    logging_strategy = 'epoch', #Логирование после каждой эпохи
    save_strategy = 'epoch')

In [13]:
class CustomTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False):
            outputs = model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                token_type_ids=inputs['token_type_ids']
            )
            loss = nn.CrossEntropyLoss()(outputs['logits'],
                                             torch.squeeze(inputs['labels']))
            return (loss, outputs) if return_outputs else loss

In [14]:
trainer = CustomTrainer(model=model.to(device),
                  tokenizer = tokenizer,
                  args = training_args,
                  train_dataset = train_dataset,
                  eval_dataset = train_dataset)

/Users/vladimirkalajcidi/anaconda3/envs/second/lib/python3.11/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False)
  warnings.warn(


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.042700,0.486878
2,0.330500,0.145684
3,0.157000,0.107445
4,0.114200,0.092553
5,0.106500,0.084216
6,0.092400,0.089199
7,0.095000,0.068764
8,0.077800,0.067575
9,0.075900,0.064136
10,0.074400,0.063654


RuntimeError: [enforce fail at inline_container.cc:595] . unexpected pos 1328594304 vs 1328594196

In [54]:
save_directory = "bert_menu"
tokenizer.save_pretrained(save_directory)
model.save_pretrained(save_directory)

In [20]:
model_entry = AutoModelForSequenceClassification.from_pretrained("bert_entry", num_labels=5)
tokenizer = BertTokenizer.from_pretrained("bert_entry")

In [55]:
test_text

array(['Сколько минут ждать очередь в 13:00 часов в музее?',
       'Сколько людей будет в 10 часов на улице?',
       'Сколько людей будет в 15 часов в цеху?',
       'Сколько людей будет в четырнадцать часов в конференц-зале?',
       'Сколько людей будет в пятнадцать часов в цеху?',
       'Сколько людей будет в 10:00 часов в аквапарке?',
       'Сколько минут ждать очередь в девять часов на вокзале?',
       'Сколько людей будет в три в спа-салоне?',
       'Сколько минут ждать очередь в девятнадцать часов в гостинице?',
       'Сколько людей будет в 17:00 часов в ботаническом саду?',
       'Сколько людей будет в двенадцать часов в цирке?',
       'Сколько людей будет в два в музее?',
       'Сколько людей будет в два в кафе?',
       'Сколько людей будет в десять часов в бассейне?',
       'Сколько минут ждать очередь в 13 часов в концертном зале?',
       'Сколько минут ждать очередь в 8 часов дня?',
       'Сколько людей будет в три на улице?',
       'Сколько минут ждать очере

In [56]:
test_labels

array([3, 1, 4, 4, 4, 1, 1, 4, 6, 5, 2, 4, 4, 1, 3, 0, 4, 5, 5, 6, 1, 5,
       6, 1])

In [23]:
def predict(text, model, tokenizer):
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_seq_len,
        return_token_type_ids=False,
        truncation=True,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt',
    )
    
    out = {
          'text': text,
          'input_ids': encoding['input_ids'].flatten(),
          'attention_mask': encoding['attention_mask'].flatten()
      }
    
    input_ids = out["input_ids"]
    attention_mask = out["attention_mask"]
    
    outputs = model(
        input_ids=input_ids.unsqueeze(0),
        attention_mask=attention_mask.unsqueeze(0)
    )

    print(outputs)
    prediction = torch.argmax(outputs.logits, dim=1).cpu().numpy()[0]

    return prediction

In [27]:
predict('какая очередь будет вечером', model_entry, tokenizer)

SequenceClassifierOutput(loss=None, logits=tensor([[ 4.7037, -0.5691, -1.6361,  0.7292, -1.9972]],
       grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)


0